<style>
/* 수업용 노트북: 기본값보다 조금 작고 촘촘하게 표시합니다. */
.jp-MarkdownOutput, .markdown-body {
    font-size: 0.94em !important;
    line-height: 1.68 !important;
}
.jp-MarkdownOutput h1, .markdown-body h1 { font-size: 1.72em !important; }
.jp-MarkdownOutput h2, .markdown-body h2 { font-size: 1.38em !important; }
.jp-MarkdownOutput h3, .markdown-body h3 { font-size: 1.14em !important; }
.jp-CodeCell, .jp-OutputArea-output, .cell.code_cell, .output_area {
    font-size: 0.92em !important;
}
.jp-MarkdownOutput table, .markdown-body table { font-size: 0.92em !important; }
.jp-MarkdownOutput blockquote, .markdown-body blockquote {
    border-left: 4px solid #4c78a8;
    padding-left: 0.9em;
    color: #4b5563;
}
</style>

# 0에서 AlexNet까지: 퍼셉트론, MLP, CNN의 출발점

이 노트북은 코드를 빠르게 실행하는 예제가 아니라, **새로운 신경망 구조가 왜 필요해졌는지**를 계산과 그림으로 확인하는 수업용 자료입니다. 단일 뉴런에서 시작해 선형 모델의 한계를 직접 확인하고, 은닉층과 합성곱이 그 한계를 어떻게 넓혀 가는지 연결합니다.

| 학습 단계 | 핵심 질문 | 확인할 결과 |
|---|---|---|
| 퍼셉트론 계산 | 하나의 뉴런은 어떻게 판단하는가? | 가중합, 편향, 임계값 |
| AND 학습 | 오답은 어떻게 가중치를 바꾸는가? | 결정경계 이동 |
| XOR 실험 | 왜 직선 하나로 풀 수 없는가? | 선형 분리의 한계 |
| MLP | 은닉층은 무엇을 추가하는가? | 비선형 경계와 특징 변환 |
| CIFAR-10 | 표 형태의 입력과 이미지는 무엇이 다른가? | 공간 구조와 합성곱의 필요성 |

> 권장 방법: 위에서부터 한 셀씩 실행하고, 그래프가 나오면 바로 다음 셀로 넘어가지 말고 본문의 **관찰 포인트**를 먼저 확인하세요.

## 학습 목표와 준비 사항

이 노트북을 마치면 다음을 자신의 말로 설명할 수 있어야 합니다.

1. 입력, 가중치, 편향, 활성화 함수가 각각 어떤 역할을 하는지 설명합니다.
2. 퍼셉트론 학습 규칙이 오답을 이용해 결정경계를 움직이는 과정을 추적합니다.
3. AND는 학습하지만 XOR은 학습하지 못하는 이유를 기하학적으로 설명합니다.
4. MLP의 은닉층이 입력 공간을 새로운 특징 공간으로 바꾸는 과정을 해석합니다.
5. 이미지에서 MLP보다 CNN이 유리한 이유를 공간 구조와 가중치 공유 관점에서 설명합니다.

사전 지식은 Python의 변수와 반복문 정도면 충분합니다. 수식은 암기 대상이 아니라 그림과 코드가 무엇을 계산하는지 확인하기 위한 지도처럼 사용합니다. 전체 실습은 CPU에서도 짧게 실행되도록 구성되어 있습니다.

In [ ]:
# 이 노트북에서 사용할 작은 학습 알고리즘과 시각화 함수입니다.
# 모든 예제는 CPU에서 수 초 안에 실행되도록 작게 설계했습니다.
import torch
import matplotlib.pyplot as plt

from cifar10_lab import (
    ClassicPerceptron,
    create_model,
    detect_environment,
    evaluate_model_detailed,
    get_lab_paths,
    load_cifar10_data,
    make_logic_gate,
    set_global_seed,
    train_model,
    train_tiny_mlp,
)
from cifar10_lab.foundation_visualization import (
    plot_activation_functions,
    plot_hidden_representation,
    plot_mlp_learning_history,
    plot_perceptron_anatomy,
    plot_perceptron_learning,
    plot_xor_comparison,
)
from cifar10_lab.visualization import plot_test_results, plot_training_history

set_global_seed(42, deterministic=True)
print('PyTorch:', torch.__version__)
print('Runtime:', detect_environment())

## 1. 퍼셉트론 한 개의 계산

퍼셉트론은 여러 입력을 받아 하나의 점수로 요약한 다음, 그 점수가 기준을 넘는지 판단합니다. 입력이 $x_1, x_2$, 가중치가 $w_1, w_2$, 편향이 $b$라면 먼저 다음 가중합을 계산합니다.

$$z = x_1w_1 + x_2w_2 + b$$

- **입력 $x$**: 관찰한 특징입니다.
- **가중치 $w$**: 각 특징을 얼마나 중요하게 볼지 나타냅니다. 부호는 판단 방향도 결정합니다.
- **편향 $b$**: 입력이 모두 0이어도 남는 기준값으로, 결정경계를 평행 이동시킵니다.
- **계단 함수**: $z \ge 0$이면 1, 그렇지 않으면 0을 출력합니다.

아래 셀에서 `sample`, `weights`, `bias`를 하나씩 바꿔 보세요. 숫자를 바꿀 때는 최종 출력만 보지 말고 각 항의 기여도와 가중합이 어떻게 변하는지 확인합니다.

In [ ]:
# 입력 두 개, 가중치 두 개, 편향 하나를 직접 바꿔 보세요.
sample = (1.0, 0.5)
weights = (0.8, -0.4)
bias = 0.1

plot_perceptron_anatomy(sample=sample, weights=weights, bias=bias)
plt.show()

### 결과 읽기: 숫자에서 결정까지

그림의 각 화살표는 `입력 × 가중치`의 기여도를 나타냅니다. 양수 기여는 점수를 올리고 음수 기여는 점수를 내립니다. 모든 기여도와 편향을 더한 값이 0을 넘는 순간 출력이 1로 바뀝니다.

**작은 실험:** `bias`만 `-0.8`로 바꾸어 다시 실행해 보세요. 입력과 가중치가 같아도 판단 기준이 이동하면서 출력이 달라질 수 있습니다.

## 2. 활성화 함수가 필요한 이유

활성화 함수는 가중합을 다음 층에 전달할 값으로 변환합니다. 활성화가 모두 선형이면 층을 여러 개 쌓아도 전체 계산은 결국 하나의 선형 변환으로 합쳐집니다. **비선형 활성화가 있어야 여러 층이 굽은 경계와 복잡한 패턴을 표현할 수 있습니다.**

| 함수 | 출력 특징 | 장점 | 주의점 |
|---|---|---|---|
| Step | 0 또는 1 | 판단 과정이 직관적 | 대부분의 지점에서 미분을 학습에 쓰기 어려움 |
| Sigmoid | 0~1 | 확률처럼 해석하기 쉬움 | 큰 절댓값에서 기울기가 매우 작아짐 |
| Tanh | -1~1 | 출력이 0을 중심으로 분포 | 역시 포화 영역에서 기울기가 작아짐 |
| ReLU | 음수는 0, 양수는 그대로 | 계산이 단순하고 양수 영역의 기울기를 유지 | 음수 영역의 뉴런이 계속 0이 될 수 있음 |

AlexNet이 깊은 CNN 학습에서 ReLU를 적극적으로 사용했다는 점은 역사적으로도 중요합니다.

In [ ]:
# 활성화 함수의 출력값뿐 아니라 곡선의 기울기와 포화 구간을 비교합니다.
plot_activation_functions()
plt.show()

### 그래프 관찰 포인트

- Step은 임계값에서 갑자기 바뀝니다.
- Sigmoid와 Tanh의 양 끝이 평평해지는 부분에서는 기울기가 거의 0입니다.
- ReLU는 0보다 큰 영역에서 직선이므로 기울기가 일정합니다.

신경망 학습은 출력값뿐 아니라 **곡선의 기울기**를 이용해 가중치를 수정합니다. 따라서 활성화 함수의 모양은 학습 속도와 안정성에 직접 영향을 줍니다.

## 3. AND 게이트를 오답 수정으로 학습하기

AND의 입력은 `(0,0)`, `(0,1)`, `(1,0)`, `(1,1)` 네 개이고, 두 값이 모두 1일 때만 정답이 1입니다. 퍼셉트론은 각 표본을 예측한 뒤 오답이면 다음 규칙으로 파라미터를 수정합니다.

$$w \leftarrow w + \eta(y-\hat{y})x$$
$$b \leftarrow b + \eta(y-\hat{y})$$

여기서 $\eta$는 학습률, $y$는 정답, $\hat{y}$는 예측입니다.

1. 현재 가중치로 가중합과 예측을 계산합니다.
2. `정답 - 예측`으로 오차 방향을 구합니다.
3. 오답에 기여한 입력만큼 가중치를 이동합니다.
4. 네 표본을 여러 번 반복해서 경계가 안정되는지 확인합니다.

AND의 두 클래스는 직선 하나로 나눌 수 있으므로 충분히 반복하면 오답이 0이 됩니다.

In [ ]:
# 1) 네 가지 입력과 AND 정답을 준비합니다.
and_x, and_y = make_logic_gate('AND')

# 2) 모든 가중치가 0인 고전적 퍼셉트론을 만듭니다.
and_model = ClassicPerceptron(input_features=2, learning_rate=0.1)

# 3) 같은 네 표본을 10회 반복해서 보여 줍니다.
# history에는 표본 하나를 볼 때마다 바뀐 가중치, 편향, 오차가 기록됩니다.
and_history = and_model.fit(and_x, and_y, epochs=10)

print('입력       정답  최종예측')
for inputs, target, prediction in zip(and_x, and_y, and_model.predict(and_x)):
    print(f'{inputs.tolist()}    {int(target)}       {int(prediction)}')
print('최종 가중치:', and_model.weights.tolist())
print('최종 편향:', and_model.bias)

### 학습 기록 해석하기

출력되는 epoch별 오답 수가 줄어드는지 확인하세요. 가중치와 편향의 최종값 자체는 실행 초기값과 표본 순서에 따라 달라질 수 있지만, 네 점을 모두 맞히는 직선을 찾았다는 사실이 핵심입니다. 서로 다른 파라미터도 같은 AND 경계를 만들 수 있습니다.

In [ ]:
# 실제로 오답이 발생해 파라미터가 갱신된 순간만 골라서 보여 줍니다.
# 각 패널에서 직선의 위치가 달라지는 것이 학습 그 자체입니다.
plot_perceptron_learning(and_x, and_y, and_history)
plt.show()

### 결정경계 그림 읽기

각 패널은 오답이 발생한 직후의 상태입니다. 점의 색은 정답 클래스이고 직선은 현재 모델의 판단 기준입니다. 학습은 데이터를 이동시키는 것이 아니라 **직선의 위치와 기울기를 바꾸는 과정**입니다.

## 4. 단일 퍼셉트론의 한계: XOR

XOR은 두 입력이 서로 다를 때만 1입니다. 양성 표본 `(0,1)`, `(1,0)`과 음성 표본 `(0,0)`, `(1,1)`이 대각선으로 엇갈려 있어 **직선 하나로 두 클래스를 분리할 수 없습니다.**

이 문제는 학습률이나 epoch가 부족해서 생기는 최적화 문제가 아닙니다. 단일 퍼셉트론이 표현할 수 있는 경계 자체가 직선뿐인 **표현력의 한계**입니다.

MLP는 여러 퍼셉트론을 은닉층에 두고 비선형 활성화를 적용합니다. 각 은닉 뉴런이 서로 다른 경계를 만들고, 출력층이 이를 조합하면 XOR처럼 굽거나 여러 조각으로 이루어진 영역도 표현할 수 있습니다.

In [ ]:
xor_x, xor_y = make_logic_gate('XOR')

# 단일 퍼셉트론: 계속 학습해도 네 점을 모두 맞히지 못합니다.
xor_perceptron = ClassicPerceptron(learning_rate=0.1)
xor_perceptron.fit(xor_x, xor_y, epochs=30)

# 작은 MLP: 2차원 입력 → Tanh 은닉층 → 출력층 구조입니다.
xor_mlp, xor_history = train_tiny_mlp(
    xor_x, xor_y, epochs=500, learning_rate=0.05, seed=42
)

print('퍼셉트론 예측:', xor_perceptron.predict(xor_x).tolist())
with torch.no_grad():
    mlp_predictions = (torch.sigmoid(xor_mlp(xor_x)) >= 0.5).long()
print('MLP 예측:      ', mlp_predictions.tolist())
print('정답:          ', xor_y.long().tolist())
print('MLP 최종 정확도:', xor_history['accuracy'][-1], '%')

### 두 모델의 출력 비교

단일 퍼셉트론은 네 점 가운데 일부를 계속 틀리지만 MLP는 은닉 뉴런을 조합해 네 점을 모두 맞힐 수 있습니다. 퍼셉트론의 실패가 정상적인 실험 결과라는 점이 중요합니다.

In [ ]:
# epoch에 따라 loss와 정확도가 서로 다른 방식으로 변하는지 확인합니다.
# 손실이 내려가고 정확도가 올라가는 과정은 경사하강법이 파라미터를 찾는 흔적입니다.
plot_mlp_learning_history(xor_history)
plt.show()

### 학습 곡선 읽기

손실은 예측이 정답에서 얼마나 벗어났는지를 연속적인 숫자로 나타내고, 정확도는 맞힌 표본의 비율입니다. 정확도는 계단처럼 변할 수 있지만 손실은 그 사이에도 감소할 수 있어 최적화 과정을 더 세밀하게 보여 줍니다.

In [ ]:
# 동일한 XOR 데이터에 단일 직선과 MLP의 비선형 경계를 나란히 그립니다.
# 왼쪽은 직선 하나, 오른쪽은 여러 은닉 뉴런이 합쳐 만든 비선형 영역입니다.
plot_xor_comparison(xor_perceptron, xor_mlp, xor_x, xor_y)
plt.show()

In [ ]:
# 은닉층은 단순히 분류만 하는 것이 아니라 입력을 분류하기 쉬운 좌표로 다시 표현합니다.
# 오른쪽 그림에서 h1, h2는 사람이 지정한 특징이 아니라 학습으로 만들어진 특징입니다.
plot_hidden_representation(xor_mlp, xor_x, xor_y)
plt.show()

### 은닉 표현의 의미

왼쪽 입력 공간에서는 XOR 점들이 대각선으로 섞여 있습니다. 은닉층을 통과한 오른쪽 공간에서는 점들의 좌표가 바뀌어 출력층이 더 단순한 경계로 나눌 수 있습니다. 딥러닝의 핵심은 정답을 외우는 것뿐 아니라 **분류하기 좋은 표현을 함께 학습하는 것**입니다.

## 5. 논리 게이트에서 CIFAR-10 이미지로

논리 게이트의 입력은 숫자 두 개뿐이지만 CIFAR-10 한 장은 `3×32×32 = 3,072`개의 값입니다. 더 중요한 차이는 픽셀이 단순한 숫자 목록이 아니라 **위치와 이웃 관계를 가진다**는 점입니다.

| 모델 | 입력을 다루는 방식 | 얻는 능력 | 남는 한계 |
|---|---|---|---|
| 퍼셉트론 | 모든 픽셀을 펼쳐 한 번에 연결 | 클래스별 선형 경계 | 비선형 패턴과 위치 관계를 표현하기 어려움 |
| MLP | 펼친 픽셀을 여러 은닉층에 통과 | 비선형 경계 | 가까운 픽셀의 관계를 명시적으로 활용하지 않음 |
| AlexNet | 작은 필터를 이미지 전체에 공유 | 경계→질감→부분 형태의 계층적 특징 | 파라미터와 계산량이 비교적 큼 |
| ResNet | 합성곱 블록에 지름길 연결 추가 | 깊은 모델의 안정적 학습 | 구조와 학습 과정이 더 복잡함 |

다음 셀은 네 모델이 같은 입력을 받아 모두 10개 클래스 점수를 출력하는지 확인합니다. 출력 모양이 같아도 내부 계산 방식과 귀납적 편향은 서로 다릅니다.

In [ ]:
# 같은 CIFAR-10 입력이 세 시대의 모델을 모두 통과하는지 확인합니다.
# 파라미터 수가 커진다고 항상 효율적이거나 정확한 것은 아닙니다.
sample_images = torch.randn(2, 3, 32, 32)

for model_id in ('perceptron', 'mlp', 'alexnet', 'resnet18'):
    model = create_model(model_id, num_classes=10, image_size=32).eval()
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    with torch.no_grad():
        output = model(sample_images)
    print(f'{model_id:12s} | parameters={parameter_count:>11,} | output={tuple(output.shape)}')

## 6. 같은 파이프라인에서 CIFAR-10을 직접 학습하기

아래 셀은 지금까지 배운 모델을 실제 CIFAR-10 분류 파이프라인에 연결합니다. 처음에는 `perceptron`, 2,048개 학습 이미지, 1 epoch로 전체 흐름을 빠르게 확인합니다.

실행 순서는 다음과 같습니다.

1. 실행 장치와 저장 폴더를 결정합니다.
2. CIFAR-10을 Train, Validation, Test로 분리합니다.
3. 전역 seed를 고정해 모델 초기값과 데이터 순서를 재현합니다.
4. 선택한 모델을 만들고 Train 데이터로 가중치를 갱신합니다.
5. Validation 정확도가 가장 높은 상태를 체크포인트에 저장합니다.
6. 학습에 사용하지 않은 Test 데이터로 마지막 성능을 측정합니다.

`CIFAR_MODEL_ID`만 `mlp`, `alexnet`, `resnet18` 순으로 바꾸면 데이터 조건을 유지한 채 구조 차이를 비교할 수 있습니다. 1 epoch 결과는 최종 성능이 아니라 **파이프라인이 정상 작동하는지 확인하는 빠른 실험**입니다.

In [ ]:
# 처음에는 perceptron으로 전체 흐름을 빠르게 확인하세요.
# 이후 mlp, alexnet, resnet18 순서로 바꾸어 차이를 관찰할 수 있습니다.
CIFAR_MODEL_ID = 'perceptron'
QUICK_EPOCHS = 1

runtime = detect_environment()
paths = get_lab_paths(create=True)
trainloader, valloader, testloader, classes = load_cifar10_data(
    batch_size=64,
    seed=42,
    num_workers=runtime.num_workers,
    pin_memory=runtime.pin_memory,
    max_train_samples=2048,
    max_val_samples=512,
    max_test_samples=512,
)

model = create_model(CIFAR_MODEL_ID, num_classes=10, image_size=32).to(runtime.device)
history = train_model(
    model,
    trainloader,
    valloader,
    runtime.device,
    epochs=QUICK_EPOCHS,
    learning_rate=0.001,
    model_id=CIFAR_MODEL_ID,
    weight_dir=paths.checkpoints_dir / 'foundations',
    experiment_id=f'{CIFAR_MODEL_ID}-tutorial',
)
result = evaluate_model_detailed(model, testloader, runtime.device, num_classes=10)
print(f'Test accuracy: {result["accuracy"]:.2f}%')

### Quick run 결과를 해석하는 기준

- Train 정확도가 오르면 가중치 갱신이 작동하고 있다는 뜻입니다.
- Validation 정확도는 학습에 직접 쓰지 않은 데이터에서 일반화 정도를 확인합니다.
- Test 정확도는 모델 선택이 끝난 뒤 마지막에 확인합니다.
- 작은 데이터와 1 epoch 결과는 모델 순위를 확정하기에 충분하지 않습니다.

공정한 비교를 하려면 모델 이외의 seed, 데이터 분할, epoch, 학습률을 동일하게 유지해야 합니다.

In [ ]:
# 그래프를 닫기 전에 Train/Validation 간격과 대각선 밖의 오분류를 확인합니다.
# 학습 곡선은 최적화가 진행되는 과정, confusion matrix는 클래스별 실수를 보여 줍니다.
plot_training_history(history)
plot_test_results(result, classes)
plt.show()

## 7. AlexNet에서 다음 시대로

AlexNet을 관찰할 때는 정확도 하나보다 개념이 어떻게 이어지는지 보세요.

- 퍼셉트론의 가중합과 활성화는 합성곱 필터의 각 위치에서도 반복됩니다.
- MLP에서 사용한 ReLU는 AlexNet의 깊은 비선형 표현을 가능하게 합니다.
- 합성곱은 같은 필터를 모든 위치에 공유해 공간 구조를 보존하고 파라미터를 절약합니다.
- Pooling은 공간 크기를 줄이며 강한 반응이나 평균 경향을 요약합니다.
- ResNet의 지름길 연결은 더 깊은 네트워크로 확장할 때 기울기 전달을 돕습니다.

### 스스로 확인할 질문

1. 편향만 바꾸면 결정경계의 기울기와 위치 중 무엇이 바뀌나요?
2. XOR을 단일 퍼셉트론이 풀지 못하는 이유를 학습 부족과 구분해 설명할 수 있나요?
3. 이미지 픽셀을 펼치면 어떤 공간 정보가 모델 구조에서 사라지나요?
4. Quick run의 Test 정확도를 모델의 최종 성능이라고 해석하면 안 되는 이유는 무엇인가요?

다음 노트북인 `cnn_internals.ipynb`에서는 AlexNet 내부의 특징맵, pooling, 수용영역을 층별로 관찰합니다.